In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("dataset.csv", engine='python')

In [ ]:
df.columns

Index(['Reviewer Name', 'Profile Link', 'Country', 'Review Count',
       'Review Date', 'Rating', 'Review Title', 'Review Text',
       'Date of Experience'],
      dtype='object')

In [ ]:
# Combine title + review text
# df['text'] = df['Review Title'].fillna('') + " " + df['Review Text'].fillna('')
# Combine review title and review text
df['text'] = df['Review Title'].fillna('') + " " + df['Review Text'].fillna('')

# Convert text to lowercase
df['text'] = df['text'].str.lower()

# Remove punctuation
df['text'] = df['text'].str.replace(r'[^\w\s]', '', regex=True)
# Check result
print(df[['text']].head())

                                                text
0  a store that doesnt want to sell anything i re...
1  had multiple orders one turned up and had mult...
2  i informed these reprobates i informed these r...
3  advertise one price then increase it on websit...
4  if i could give a lower rate i would if i coul...


In [ ]:
df.head(5)

,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience,text
0,Eugene ath,/users/66e8185ff1598352d6b3701a,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...","September 16, 2024",a store that doesnt want to sell anything i re...
1,Daniel ohalloran,/users/5d75e460200c1f6a6373648c,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,"September 16, 2024",had multiple orders one turned up and had mult...
2,p fisher,/users/546cfcf1000064000197b88f,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,"September 16, 2024",i informed these reprobates i informed these r...
3,Greg Dunn,/users/62c35cdbacc0ea0012ccaffa,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,"September 17, 2024",advertise one price then increase it on websit...
4,Sheila Hannah,/users/5ddbe429478d88251550610e,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,"September 16, 2024",if i could give a lower rate i would if i coul...


In [ ]:
df.isnull().sum()

,0
Reviewer Name,0
Profile Link,51
Country,160
Review Count,159
Review Date,159
Rating,159
Review Title,159
Review Text,159
Date of Experience,267
text,0


In [ ]:
# Extract number from rating text
# Extract rating number safely
df['Rating'] = df['Rating'].astype(str).str.extract(r'(\d+)')

# Convert to numeric (handle errors safely)
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

# Drop rows where rating couldn't be extracted
df = df.dropna(subset=['Rating'])

# Convert to int
df['Rating'] = df['Rating'].astype(int)

# Check
print(df['Rating'].head())
print(df['Rating'].dtype)

0    1
1    1
2    1
3    1
4    1
Name: Rating, dtype: int64
int64


In [ ]:
def get_sentiment(rating):
    if rating <= 2:
        return "Negative"
    elif rating == 3:
        return "Neutral"
    else:
        return "Positive"

df['sentiment'] = df['Rating'].apply(get_sentiment)

# Check
print(df[['Rating', 'sentiment']].tail())

       Rating sentiment
21209       5  Positive
21210       5  Positive
21211       3   Neutral
21212       5  Positive
21213       4  Positive


In [ ]:
# ===========================
# Issue Keywords
# ===========================

issue_keywords = {
    "Refund": [
        "refund",
        "return",
        "money back",
        "return my money",
        "refund request",
        "cancel order"
    ],

    "Delivery": [
        "delivery",
        "late",
        "delay",
        "delayed",
        "late delivery",
        "delivery delayed",
        "not delivered",
        "not been delivered"
        "still waiting",
        "never arrived",
        "order not received",
        "shipment delayed",
        "missing package"
    ],

    "Product Issue": [
        "broken",
        "damaged",
        "defective",
        "not working",
        "faulty",
        "cracked",
        "stopped working",
        "poor quality",
        "missing parts",
        "battery issue"
    ],

    "Account Issue": [
        "login",
        "account",
        "password",
        "sign in",
        "unable to login",
        "can't login",
        "account locked",
        "reset password"
    ],

    "Billing": [
        "payment",
        "charged",
        "charged twice",
        "double charged",
        "billing",
        "billing issue",
        "payment failed",
        "money deducted",
        "incorrect bill",
        "wrong amount",
        "extra charge"
    ]
}


# ===========================
# Issue Detection Function
# ===========================

def get_issue(text):

    text = str(text).lower()

    for issue, keywords in issue_keywords.items():
        if any(keyword in text for keyword in keywords):
            return issue

    return "General"


# Create Issue Column
df["issue"] = df["text"].apply(get_issue)

# Check Results
print(df[["text", "issue"]].head())

                                                text          issue
0  a store that doesnt want to sell anything i re...  Account Issue
1  had multiple orders one turned up and had mult...       Delivery
2  i informed these reprobates i informed these r...  Account Issue
3  advertise one price then increase it on websit...        General
4  if i could give a lower rate i would if i coul...         Refund


In [ ]:
# def get_priority(row):
#     sentiment = row['sentiment']
#     text = row['text'].lower()

#     # High priority conditions
#     if sentiment == "Negative" and (
#         "urgent" in text or "immediately" in text or "now" in text or "worst" in text
#     ):
#         return "High"

#     # Medium priority
#     elif sentiment == "Negative":
#         return "Medium"

#     # Neutral
#     elif sentiment == "Neutral":
#         return "Medium"

#     # Positive
#     else:
#         return "Low"


# df['priority'] = df.apply(get_priority, axis=1)

# # Check
# print(df[['sentiment', 'priority']].head())

In [ ]:
def get_priority(row):

    sentiment = row["sentiment"]
    issue = row["issue"]
    text = str(row["text"]).lower()

    urgent_words = [
        "urgent",
        "immediately",
        "asap",
        "worst",
        "fraud",
        "legal",
        "lawsuit"
    ]

    if sentiment == "Positive":
        return "Low"

    if sentiment == "Neutral":
        return "Medium"

    # Negative reviews
    if issue == "Refund":
        return "High"

    if issue == "Billing":
        return "High"

    if any(word in text for word in urgent_words):
        return "High"

    return "Medium"


df["priority"] = df.apply(get_priority, axis=1)

In [ ]:
# Input feature (customer reviews)
reviews = df['text']

# Multiple target columns
targets = df[['sentiment', 'issue', 'priority']]

In [ ]:
from sklearn.model_selection import train_test_split

reviews_train, reviews_test, targets_train, targets_test = train_test_split(
    reviews,
    targets,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,3),
    stop_words="english",
    min_df=2
)

reviews_train = vectorizer.fit_transform(reviews_train)
reviews_test = vectorizer.transform(reviews_test)

In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import LinearSVC

svc_model = MultiOutputClassifier(
    LinearSVC(class_weight='balanced')
)

svc_model.fit(reviews_train, targets_train)

# Predictions
svc_predictions = svc_model.predict(reviews_test)

In [ ]:
from sklearn.metrics import classification_report

print("Sentiment Report")
print(classification_report(
    targets_test['sentiment'],
    svc_predictions[:, 0]
))

print("Issue Report")
print(classification_report(
    targets_test['issue'],
    svc_predictions[:, 1]
))

print("Priority Report")
print(classification_report(
    targets_test['priority'],
    svc_predictions[:, 2]
))

Sentiment Report
              precision    recall  f1-score   support

    Negative       0.94      0.95      0.94      2921
     Neutral       0.25      0.19      0.22       166
    Positive       0.87      0.88      0.88      1124

    accuracy                           0.90      4211
   macro avg       0.69      0.67      0.68      4211
weighted avg       0.89      0.90      0.90      4211

Issue Report
               precision    recall  f1-score   support

Account Issue       0.80      0.84      0.82       286
      Billing       0.62      0.41      0.49        49
     Delivery       0.92      0.87      0.89       964
      General       0.91      0.98      0.94      1713
Product Issue       0.66      0.56      0.60        72
       Refund       0.96      0.90      0.93      1127

     accuracy                           0.91      4211
    macro avg       0.81      0.76      0.78      4211
 weighted avg       0.91      0.91      0.91      4211

Priority Report
              precis

In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Model
lr_model = MultiOutputClassifier(
    LogisticRegression(class_weight='balanced', max_iter=1000)
)

# Train
lr_model.fit(reviews_train, targets_train)

# Predict
lr_predictions = lr_model.predict(reviews_test)

# Evaluation
print("Sentiment Report")
print(classification_report(
    targets_test['sentiment'],
    lr_predictions[:, 0]
))

print("Issue Report")
print(classification_report(
    targets_test['issue'],
    lr_predictions[:, 1]
))

print("Priority Report")
print(classification_report(
    targets_test['priority'],
    lr_predictions[:, 2]
))

Sentiment Report
              precision    recall  f1-score   support

    Negative       0.95      0.91      0.93      2921
     Neutral       0.20      0.39      0.27       166
    Positive       0.88      0.86      0.87      1124

    accuracy                           0.88      4211
   macro avg       0.68      0.72      0.69      4211
weighted avg       0.90      0.88      0.89      4211

Issue Report
               precision    recall  f1-score   support

Account Issue       0.74      0.91      0.82       286
      Billing       0.48      0.67      0.56        49
     Delivery       0.90      0.83      0.86       964
      General       0.91      0.96      0.93      1713
Product Issue       0.50      0.69      0.58        72
       Refund       0.93      0.82      0.87      1127

     accuracy                           0.88      4211
    macro avg       0.74      0.81      0.77      4211
 weighted avg       0.89      0.88      0.88      4211

Priority Report
              precis

In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

# Model
dt_model = MultiOutputClassifier(
    DecisionTreeClassifier(random_state=42)
)

# Train
dt_model.fit(reviews_train, targets_train)

# Predict
dt_predictions = dt_model.predict(reviews_test)

# Evaluation
print("Sentiment Report")
print(classification_report(
    targets_test['sentiment'],
    dt_predictions[:, 0]
))

print("Issue Report")
print(classification_report(
    targets_test['issue'],
    dt_predictions[:, 1]
))

print("Priority Report")
print(classification_report(
    targets_test['priority'],
    dt_predictions[:, 2]
))

Sentiment Report
              precision    recall  f1-score   support

    Negative       0.90      0.87      0.89      2921
     Neutral       0.15      0.13      0.14       166
    Positive       0.72      0.78      0.75      1124

    accuracy                           0.82      4211
   macro avg       0.59      0.59      0.59      4211
weighted avg       0.82      0.82      0.82      4211

Issue Report
               precision    recall  f1-score   support

Account Issue       0.92      0.94      0.93       286
      Billing       0.76      0.86      0.81        49
     Delivery       0.94      0.93      0.93       964
      General       0.96      0.97      0.96      1713
Product Issue       0.75      0.74      0.74        72
       Refund       0.95      0.94      0.95      1127

     accuracy                           0.94      4211
    macro avg       0.88      0.89      0.89      4211
 weighted avg       0.94      0.94      0.94      4211

Priority Report
              precis

In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Model
rf_model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'
    )
)

# Train
rf_model.fit(reviews_train, targets_train)

# Predict
rf_predictions = rf_model.predict(reviews_test)

# Evaluation
print("Sentiment Report")
print(classification_report(
    targets_test['sentiment'],
    rf_predictions[:, 0]
))

print("Issue Report")
print(classification_report(
    targets_test['issue'],
    rf_predictions[:, 1]
))

print("Priority Report")
print(classification_report(
    targets_test['priority'],
    rf_predictions[:, 2]
))

Sentiment Report
              precision    recall  f1-score   support

    Negative       0.90      0.96      0.93      2921
     Neutral       0.17      0.01      0.02       166
    Positive       0.84      0.82      0.83      1124

    accuracy                           0.89      4211
   macro avg       0.64      0.60      0.59      4211
weighted avg       0.86      0.89      0.87      4211

Issue Report
               precision    recall  f1-score   support

Account Issue       0.88      0.92      0.90       286
      Billing       0.82      0.18      0.30        49
     Delivery       0.95      0.87      0.91       964
      General       0.89      1.00      0.94      1713
Product Issue       1.00      0.21      0.34        72
       Refund       0.95      0.91      0.93      1127

     accuracy                           0.92      4211
    macro avg       0.91      0.68      0.72      4211
 weighted avg       0.92      0.92      0.91      4211

Priority Report
              precis

# New Section

In [ ]:
review = "Resetting my password didn't help. I still can't sign in."

review_vector = vectorizer.transform([review])
prediction = svc_model.predict(review_vector)

print(prediction)

[['Negative' 'Account Issue' 'Medium']]


In [ ]:
# import joblib

# # Save model
# joblib.dump(svc_model, 'linear_svc_model.pkl')

# # Save vectorizer
# joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

# print("Files saved successfully!")

In [ ]:
# from google.colab import files

# # Download model
# files.download('linear_svc_model.pkl')

# # Download vectorizer
# files.download('tfidf_vectorizer.pkl')